# Check the report - EP_0001

Read the claims beside the evidence, note anything wrong, edit the copy below, then save it to public.
The original model response stays in the attempt folder. No model calls here.

In [1]:
import copy
import json
import sys
from decimal import Decimal
from pathlib import Path
from pprint import pprint

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/benchmark/private/packets").is_dir())
sys.path.insert(0, str(repo))
from src.benchmark.prepare import read, save, mask, assert_masked, check_report
from src.benchmark.schemas import Packet, Report

folder = repo / "data/benchmark"
attempt_id = "fa4c7c8cf694408e84a615d6b28ac489"
attempt_folder = folder / "private/attempts" / attempt_id
attempt = read(attempt_folder / "attempt.json")
sent_packet = json.loads(attempt["request"]["messages"][1]["content"])["evidence_packet"]
original_report = read(attempt_folder / "candidate.json")
aliases = read(folder / "private/aliases.json")
packet = mask(sent_packet, aliases)
report = copy.deepcopy(original_report)
print("Packet:", packet["packet_id"], "| Model:", attempt["request"]["model"])
print("Packet masking changed:", packet != sent_packet)

Packet: EP_0001 | Model: google/gemini-2.5-flash
Packet masking changed: True


In [2]:
evidence = {e["evidence_id"]: e["text"] for e in packet["evidence"]}
for claim in report["claims"]:
    print("\nCLAIM", claim["claim_id"], claim["text"])
    for evidence_id in claim["evidence_ids"]:
        print("EVIDENCE", evidence_id, evidence.get(evidence_id, "MISSING"))
print("\nRationale:", report["rationale"])


CLAIM C1 Total net sales for COMPANY_01 increased by 6.43% from $391,035 million in fiscal year 2024 to $416,161 million in fiscal year 2025 (Item 7, EP_0001_E01).
EVIDENCE EP_0001_E01 Products and Services Performance
The following table shows net sales by category for 2025, 2024 and 2023 (dollars in millions):
2025 Change 2024 Change 2023
PRODUCT_01 $ 209,586 4 % $ 201,183 — % $ 200,583 
PRODUCT_06 33,708 12 % 29,984 2 % 29,357 
PRODUCT_05 28,023 5 % 26,694 (6)% 28,300 
Wearables, Home and Accessories 35,686 (4)% 37,005 (7)% 39,845 
Services 109,158 14 % 96,169 13 % 85,200 
Total net sales $ 416,161 6 % $ 391,035 2 % $ 383,285 
(1) Services net sales include amortization of the deferred value of services bundled in the sales price of certain products.

CLAIM C2 The Services sales category experienced a 13.51% increase in net sales, rising from $96,169 million in fiscal year 2024 to $109,158 million in fiscal year 2025 (Item 7, EP_0001_E01). This growth was primarily driven by higher

## Numbers

Compare the claimed amount or percentage with the calculation. Also read the prose above:
these checks only cover the structured assertion, not every number in a sentence.

In [3]:
facts_by_id = {f["fact_id"]: f for f in packet["facts"]}
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        print(claim["claim_id"], "Check against the excerpt above.")
        continue
    ids = [spec.get("fact_id")] if spec["kind"] == "numeric" else [spec.get("from_fact_id"), spec.get("to_fact_id")]
    periods = [spec["period"]] if spec["kind"] == "numeric" else [spec["from_period"], spec["to_period"]]
    if all(ids):
        selected = [facts_by_id.get(fact_id) for fact_id in ids]
        print(claim["claim_id"], "Referenced facts:")
        pprint(selected)
        if any(f is None for f in selected):
            print("Unknown fact ID; check manually.")
            continue
    else:
        print(claim["claim_id"], "Older candidate: references are added in the edit cell below.")
        matches = [[f for f in packet["facts"] if f["metric"] == spec["metric"]
                    and f["evidence_id"] in claim["evidence_ids"] and f["period"] == period
                    and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])] for period in periods]
        if any(len(found) != 1 for found in matches):
            print("No exact fact match; check manually.")
            continue
        selected = [found[0] for found in matches]
        pprint(selected)
    values = [Decimal(f["value"]) for f in selected]
    if spec["kind"] == "percentage_change" and values[0] == 0:
        print(claim["claim_id"], "Cannot calculate percentage change from zero.")
        continue
    expected = values[0] if spec["kind"] == "numeric" else (values[1] - values[0]) / values[0] * 100
    print(claim["claim_id"], "claimed:", spec["value"], "calculated:", expected,
          "rounded:", expected.quantize(Decimal("0.01")))
pprint(check_report(Packet.model_validate(packet), Report.model_validate(report)))


C1 Older candidate: references are added in the edit cell below.
[{'evidence_id': 'EP_0001_E01',
  'fact_id': 'EP_0001_F02',
  'metric': 'net_sales',
  'period': {'end': '2024-09-28', 'label': 'FY2024', 'start': '2023-10-01'},
  'unit': 'USD_million',
  'value': '391035'},
 {'evidence_id': 'EP_0001_E01',
  'fact_id': 'EP_0001_F01',
  'metric': 'net_sales',
  'period': {'end': '2025-09-27', 'label': 'FY2025', 'start': '2024-09-29'},
  'unit': 'USD_million',
  'value': '416161'}]
C1 claimed: 6.43 calculated: 6.425511782832738757400232716 rounded: 6.43
C2 Older candidate: references are added in the edit cell below.
[{'evidence_id': 'EP_0001_E01',
  'fact_id': 'EP_0001_F04',
  'metric': 'services_net_sales',
  'period': {'end': '2024-09-28', 'label': 'FY2024', 'start': '2023-10-01'},
  'unit': 'USD_million',
  'value': '96169'},
 {'evidence_id': 'EP_0001_E01',
  'fact_id': 'EP_0001_F03',
  'metric': 'services_net_sales',
  'period': {'end': '2025-09-27', 'label': 'FY2025', 'start': '2024-

## Notes / feedback

- C1: sales growth is 6.43%. Amounts and years match E01.
- C2: Services growth is 13.51%. The reasons given match E02.
- C3: the sales decline is 3.56%. Both the numbers and explanation match E01/E02.
- C4: the overseas sales, suppliers and economic risks match E03.
- The original excerpts match PDF pages 27 and 8. The PDF hash matches the source record.
- The report includes the declining category and economic risks, rather than just the positive results.
- HOLD is reasonable from this limited evidence; it is not a guaranteed investment outcome.
- Earlier feedback corrected 6.42% to 6.43% and changed 'segment' to 'sales category'. The kept candidate already has these fixes.
- No report edits needed. Company names use aliases; figures and descriptive category names can still give clues.
- Decision: ready to use as the clean report.
- Added explicit fact IDs to the reviewed report copy. Amounts, periods and wording are unchanged by this citation update; the original model response is kept.

In [4]:
# Make any corrections to report here. Keep the original candidate unchanged.

# Add explicit source-fact references to the reviewed copy of older candidates.
# This does not change the saved model response, numbers, or wording.
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        continue
    fields = [("fact_id", "period")] if spec["kind"] == "numeric" else [("from_fact_id", "from_period"), ("to_fact_id", "to_period")]
    for id_field, period_field in fields:
        if spec.get(id_field):
            continue
        matches = [f for f in packet["facts"] if f["metric"] == spec["metric"]
                   and f["period"] == spec[period_field] and f["evidence_id"] in claim["evidence_ids"]
                   and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])]
        assert len(matches) == 1, "Check the fact mapping manually: " + claim["claim_id"]
        spec[id_field] = matches[0]["fact_id"]

pprint(report, sort_dicts=False)

{'schema_version': '0.1',
 'report_id': 'R_fa4c7c8cf694408e84a615d6b28ac489',
 'recommendation': 'HOLD',
 'claims': [{'claim_id': 'C1',
             'text': 'Total net sales for COMPANY_01 increased by 6.43% from '
                     '$391,035 million in fiscal year 2024 to $416,161 million '
                     'in fiscal year 2025 (Item 7, EP_0001_E01).',
             'evidence_ids': ['EP_0001_E01'],
             'check_spec': {'kind': 'percentage_change',
                            'metric': 'net_sales',
                            'from_period': {'label': 'FY2024',
                                            'start': '2023-10-01',
                                            'end': '2024-09-28'},
                            'to_period': {'label': 'FY2025',
                                          'start': '2024-09-29',
                                          'end': '2025-09-27'},
                            'value': '6.43',
                            'unit': 'percent',
     

## Save the final copy

Set `ready = True` after the checks and edits are finished. This writes the final report and
masked packet to public. It does not change the raw attempt or call a model.
Save this notebook in `private/reviews/` using `<packet_id>_review.ipynb`.
One notebook contains the check, feedback, edits and output; no separate review JSON is needed.

The report is saved as `public/reports/<packet_id>/clean.json`.

In [5]:
ready = True
if ready:
    final_report = Report.model_validate(report)
    final_packet = Packet.model_validate(packet)
    checks = check_report(final_packet, final_report, require_fact_ids=True)
    assert not checks["errors"], checks["errors"]
    assert_masked(report, aliases)
    assert_masked(packet, aliases)
    save(folder / "public/evidence_packets" / (packet["packet_id"] + ".json"), final_packet.model_dump(mode="json"))
    output = folder / "public/reports" / packet["packet_id"] / "clean.json"
    save(output, final_report.model_dump(mode="json"))
    print("Saved:", output)
else:
    print("Not saved yet.")

Saved: /Users/claudia/Desktop/dsa4213/dsa4213 proj/DSA4213_Group11/data/benchmark/public/reports/EP_0001/clean.json
